In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from collections import Counter, defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool
import networkx as nx
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data
train_data = pd.read_csv("/home/info-sec-lab/Downloads/Aman/Codelite/train_original.csv")
test_data = pd.read_csv("/home/info-sec-lab/Downloads/Aman/Codelite/test_original.csv")

tag_vocab = pd.read_csv("/home/info-sec-lab/Downloads/Aman/Codelite/unique_tags.csv")["Tag"].tolist()

label_encoder = LabelEncoder()
label_encoder.fit(tag_vocab)

train_data["label_id"] = label_encoder.transform(train_data["language"])
test_data["label_id"] = label_encoder.transform(test_data["language"])

print(f"Dataset has {len(train_data)} training samples and {len(test_data)} test samples")
print(f"Number of languages: {len(tag_vocab)}")

In [ ]:
# Language to parser mapping
LANGUAGE_MAPPING = {
    'bash': 'bash',
    'c': 'c',
    'c#': 'c_sharp',
    'c++': 'cpp',
    'css': 'css',
    'haskell': 'haskell',
    'html': 'html',
    'java': 'java',
    'javascript': 'javascript',
    'lua': 'lua',
    'markdown': 'python',
    'objective-c': 'c',
    'perl': 'perl',
    'php': 'php',
    'python': 'python',
    'r': 'python',
    'ruby': 'ruby',
    'scala': 'scala',
    'sql': 'sql',
    'swift': 'swift',
    'vb.net': 'c_sharp',
}

In [ ]:
class MultiLanguageParser:
    def __init__(self):
        self.parsers = {}
    
    def get_parser(self, language):
        """Get or create parser for a specific language"""
        lang_key = language.lower()
        
        if lang_key not in self.parsers:
            try:
                ts_lang = LANGUAGE_MAPPING.get(lang_key, 'python')
                from tree_sitter_languages import get_parser
                self.parsers[lang_key] = get_parser(ts_lang)
            except Exception as e:
                print(f"Error creating parser for {language}: {e}")
                from tree_sitter_languages import get_parser
                self.parsers[lang_key] = get_parser("python")
        
        return self.parsers[lang_key]

parser_manager = MultiLanguageParser()


In [ ]:
class ASTGraphBuilder:
    def __init__(self):
        self.node_counter = Counter()
        self.node2idx = {"<PAD>": 0, "<UNK>": 1}
        self.edge_types = ['child', 'next_sibling']  # Edge types
        self.edge_type2idx = {et: i for i, et in enumerate(self.edge_types)}
    
    def parse_to_ast(self, code, language):
        """Parse code to AST nodes"""
        try:
            if not isinstance(code, str) or not code.strip():
                return []
            
            if len(code) > 5000:
                code = code[:5000]
            
            parser = parser_manager.get_parser(language)
            tree = parser.parse(code.encode('utf-8', errors='ignore'))
            
            if tree.root_node is None:
                return []
            
            return self._extract_nodes(tree.root_node)
        except Exception as e:
            return []
    
    def _extract_nodes(self, node, parent_id=None, nodes=None, edges=None, depth=0):
        """Extract AST nodes and edges recursively"""
        if nodes is None:
            nodes = []
        if edges is None:
            edges = []
        
        if depth > 100:
            return nodes, edges
        
        # Create node
        node_id = len(nodes)
        node_type = node.type
        nodes.append({
            'id': node_id,
            'type': node_type,
            'is_named': node.is_named,
            'depth': depth
        })
        
        # Add parent-child edge
        if parent_id is not None:
            edges.append((parent_id, node_id, 'child'))
        
        # Process children
        children = list(node.children)
        prev_child_id = None
        
        for child in children:
            if child.is_named or depth < 3:  # Include named nodes or shallow nodes
                child_nodes, child_edges = self._extract_nodes(
                    child, node_id, nodes, edges, depth + 1
                )
                nodes = child_nodes
                edges = child_edges
                
                # Add sibling edges
                if prev_child_id is not None:
                    edges.append((prev_child_id, child_nodes[-1]['id'], 'next_sibling'))
                
                prev_child_id = child_nodes[-1]['id']
        
        return nodes, edges
    
    def build_graph(self, code, language, max_nodes=300):
        """Build graph from AST"""
        nodes, edges = self.parse_to_ast(code, language)
        
        if not nodes:
            # Return empty graph
            return {
                'node_features': np.zeros((1, 1), dtype=np.float32),
                'edge_index': np.zeros((2, 0), dtype=np.long),
                'edge_attr': np.zeros((0, 1), dtype=np.float32)
            }
        
        if len(nodes) > max_nodes:
            nodes = nodes[:max_nodes]
            node_ids = set(node['id'] for node in nodes)
            edges = [(u, v, t) for u, v, t in edges if u in node_ids and v in node_ids]
        
        # Build node features
        node_features = []
        node_mapping = {}
        
        for i, node in enumerate(nodes):
            node_mapping[node['id']] = i
            feat = [
                hash(node['type']) % 1000,
                node['depth'],
                1.0 if node['is_named'] else 0.0
            ]
            node_features.append(feat)
        
        node_features = np.array(node_features, dtype=np.float32)
        
        # Build edge index and edge attributes
        edge_indices = []
        edge_attrs = []
        
        for u, v, edge_type in edges:
            if u in node_mapping and v in node_mapping:
                edge_indices.append([node_mapping[u], node_mapping[v]])
                # One-hot encoding of edge type
                edge_attr = np.zeros(len(self.edge_types), dtype=np.float32)
                edge_attr[self.edge_type2idx[edge_type]] = 1.0
                edge_attrs.append(edge_attr)
        
        if not edge_indices:
            # Add self-loops if no edges
            for i in range(len(nodes)):
                edge_indices.append([i, i])
                edge_attr = np.zeros(len(self.edge_types), dtype=np.float32)
                edge_attr[0] = 1.0  # Mark as self-loop
                edge_attrs.append(edge_attr)
        
        edge_index = np.array(edge_indices, dtype=np.long).T
        edge_attr = np.array(edge_attrs, dtype=np.float32)
        
        return {
            'node_features': node_features,
            'edge_index': edge_index,
            'edge_attr': edge_attr
        }
    
    def build_vocabulary(self, dataset, max_samples_per_lang=100):
        """Build vocabulary from dataset"""
        print("Building vocabulary from AST nodes...")
        
        for lang in dataset["language"].unique():
            lang_samples = dataset[dataset["language"] == lang].head(max_samples_per_lang)
            print(f"Processing {len(lang_samples)} samples of {lang}...")
            
            for _, row in lang_samples.iterrows():
                try:
                    nodes, _ = self.parse_to_ast(row["code"], row["language"])
                    for node in nodes:
                        self.node_counter[node['type']] += 1
                except:
                    continue
        
        # Build vocabulary
        for node_type, count in self.node_counter.items():
            if count >= 2:  # Appears at least twice
                self.node2idx[node_type] = len(self.node2idx)
        
        print(f"Vocabulary size: {len(self.node2idx)}")
        print(f"Top 20 node types: {list(self.node_counter.most_common(20))}")
        
        return self.node2idx


In [ ]:
# Initialize graph builder
graph_builder = ASTGraphBuilder()

# Build vocabulary
node2idx = graph_builder.build_vocabulary(train_data, max_samples_per_lang=50)

def encode_graph(code, language, label, max_nodes=200):
    """Encode code as graph"""
    graph_data = graph_builder.build_graph(code, language, max_nodes=max_nodes)
    
    # Convert node type hashes to indices
    node_features = graph_data['node_features']
    if len(node_features) > 0:
        # Replace hash with vocabulary index
        for i in range(len(node_features)):
            node_type_hash = int(node_features[i, 0])
            node_features[i, 0] = node_type_hash % len(node2idx)
    
    return {
        'x': torch.FloatTensor(node_features),
        'edge_index': torch.LongTensor(graph_data['edge_index']),
        'edge_attr': torch.FloatTensor(graph_data['edge_attr']),
        'y': torch.tensor(label, dtype=torch.long)
    }

class ASTGraphDataset(Dataset):
    def __init__(self, df, max_nodes=200):
        self.df = df.reset_index(drop=True)
        self.max_nodes = max_nodes
        print(f"Dataset size: {len(self.df)} samples")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        graph_data = encode_graph(
            row["code"],
            row["language"],
            row["label_id"],
            max_nodes=self.max_nodes
        )
        
        # Create PyG Data object
        data = Data(
            x=graph_data['x'],
            edge_index=graph_data['edge_index'],
            edge_attr=graph_data['edge_attr'],
            y=graph_data['y']
        )
        
        # Add batch information
        data.num_nodes = graph_data['x'].size(0)
        
        return data

class ASTGNN(nn.Module):
    def __init__(self, 
                 node_input_dim, 
                 edge_input_dim,
                 hidden_dim, 
                 num_classes,
                 num_layers=3,
                 dropout=0.3):
        super(ASTGNN, self).__init__()
        
        self.node_input_dim = node_input_dim
        self.edge_input_dim = edge_input_dim
        self.hidden_dim = hidden_dim
        
        # Node feature encoder
        self.node_encoder = nn.Sequential(
            nn.Linear(node_input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout)
        )
        
        # Edge feature encoder
        self.edge_encoder = nn.Sequential(
            nn.Linear(edge_input_dim, hidden_dim),
            nn.ReLU(),
            nn.LayerNorm(hidden_dim)
        )
        
        # GNN layers
        self.gnn_layers = nn.ModuleList()
        for i in range(num_layers):
            if i == 0:
                conv = GCNConv(hidden_dim, hidden_dim)
            else:
                conv = GCNConv(hidden_dim, hidden_dim)
            self.gnn_layers.append(conv)
        
        # Layer normalizations
        self.layer_norms = nn.ModuleList([
            nn.LayerNorm(hidden_dim) for _ in range(num_layers)
        ])
        
        # Attention pooling
        self.attention_pool = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, data, batch=None):
        x, edge_index, edge_attr = data.x, data.edge_index, data.edge_attr
        
        # Encode node features
        x = self.node_encoder(x)
        

        
        # Apply GNN layers
        for i, (conv, norm) in enumerate(zip(self.gnn_layers, self.layer_norms)):
            x_new = conv(x, edge_index)
            x_new = norm(x_new)
            x_new = F.relu(x_new)
            x_new = self.dropout(x_new)
            
            # Residual connection
            if i > 0:
                x = x + x_new
            else:
                x = x_new
        
        # Global pooling with attention
        if batch is not None:
            # Batch processing
            graph_embeddings = []
            for graph_idx in torch.unique(batch):
                mask = batch == graph_idx
                graph_nodes = x[mask]
                
                # Attention weights
                attention_scores = self.attention_pool(graph_nodes)
                attention_weights = F.softmax(attention_scores, dim=0)
                
                # Weighted sum
                weighted = attention_weights * graph_nodes
                graph_embedding = weighted.sum(dim=0)
                
                # Also include mean pooling
                mean_pool = graph_nodes.mean(dim=0)
                
                # Combine
                combined = torch.cat([graph_embedding, mean_pool], dim=0)
                graph_embeddings.append(combined)
            
            x = torch.stack(graph_embeddings)
        else:
            attention_scores = self.attention_pool(x)
            attention_weights = F.softmax(attention_scores, dim=0)
            weighted = attention_weights * x
            graph_embedding = weighted.sum(dim=0)
            
            mean_pool = x.mean(dim=0)
            x = torch.cat([graph_embedding, mean_pool], dim=0).unsqueeze(0)
        
        # Classify
        logits = self.classifier(x)
        
        return logits


In [ ]:
class BatchASTGNN(ASTGNN):
    """Wrapper for batch processing"""
    def forward(self, data):
        return super().forward(data, data.batch if hasattr(data, 'batch') else None)

def collate_fn(batch):
    """Custom collate function for PyG Data objects"""
    return Batch.from_data_list(batch)

In [ ]:
MAX_NODES = 150  
MAX_SAMPLES = 100000  
balanced_samples = []
for lang in train_data["language"].unique():
    lang_samples = train_data[train_data["language"] == lang].head(MAX_SAMPLES)
    balanced_samples.append(lang_samples)

train_df = pd.concat(balanced_samples, ignore_index=True)
test_df = test_data

train_dataset = ASTGraphDataset(train_df, max_nodes=MAX_NODES)
test_dataset = ASTGraphDataset(test_df, max_nodes=MAX_NODES)

In [ ]:
# Create dataloaders
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

In [ ]:
# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nUsing device: {device}")

sample_data = train_dataset[0]
node_input_dim = sample_data.x.size(1)
edge_input_dim = sample_data.edge_attr.size(1) if sample_data.edge_attr is not None else 1

print(f"Node feature dimension: {node_input_dim}")
print(f"Edge feature dimension: {edge_input_dim}")

model = BatchASTGNN(
    node_input_dim=node_input_dim,
    edge_input_dim=edge_input_dim,
    hidden_dim=128,
    num_classes=len(tag_vocab),
    num_layers=3,
    dropout=0.3
).to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# Optimizer and scheduler
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=3, factor=0.5
)

criterion = nn.CrossEntropyLoss()

# Training function
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    
    for batch_idx, data in enumerate(loader):
        data = data.to(device)
        optimizer.zero_grad()
        
        logits = model(data)
        loss = criterion(logits, data.y)
        
        # Check for NaN
        if torch.isnan(loss):
            print(f"NaN loss detected at batch {batch_idx}, skipping...")
            continue
        
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        preds = logits.argmax(dim=1)
        correct = (preds == data.y).sum().item()
        total_correct += correct
        total_samples += data.y.size(0)
        
        if batch_idx % 20 == 0:
            batch_acc = correct / data.y.size(0) if data.y.size(0) > 0 else 0
            print(f"  Batch {batch_idx:3d}, Loss: {loss.item():.4f}, Acc: {batch_acc:.4f}")
    
    avg_loss = total_loss / len(loader) if len(loader) > 0 else 0
    accuracy = total_correct / total_samples if total_samples > 0 else 0
    
    return avg_loss, accuracy

# Evaluation function
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_samples = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            
            logits = model(data)
            loss = criterion(logits, data.y)
            
            total_loss += loss.item()
            
            preds = logits.argmax(dim=1)
            correct = (preds == data.y).sum().item()
            total_correct += correct
            total_samples += data.y.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(data.y.cpu().numpy())
    
    avg_loss = total_loss / len(loader) if len(loader) > 0 else 0
    accuracy = total_correct / total_samples if total_samples > 0 else 0
    
    return avg_loss, accuracy, all_preds, all_labels

In [ ]:
# Training loop
print("\nStarting training...")
NUM_EPOCHS = 10
best_val_acc = 0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
    
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, optimizer, criterion, device
    )
    
    # Evaluate
    val_loss, val_acc, val_preds, val_labels = evaluate(
        model, test_loader, criterion, device
    )
    
    # Update learning rate
    scheduler.step(val_loss)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'node2idx': node2idx,
            'label_encoder': label_encoder,
        }, 'best_ast_gnn_model.pth')
        print(f"  Saved best model with accuracy: {val_acc:.4f}")


In [ ]:
print("\n" + "="*50)
print("Final Evaluation")
print("="*50)

model.eval()
test_loss, test_acc, test_preds, test_labels = evaluate(
    model, test_loader, criterion, device
)

print(f"\nTest Accuracy: {test_acc:.4f}")

try:
    model.load_state_dict(torch.load('best_ast_gnn_model_weights.pth', weights_only=False))
    print("Loaded best model weights")
    
    test_loss, test_acc, test_preds, test_labels = evaluate(
        model, test_loader, criterion, device
    )
    print(f"Best Model Test Accuracy: {test_acc:.4f}")
except Exception as e:
    print(f"Could not load best model, using current model: {e}")

pred_languages = label_encoder.inverse_transform(test_preds)
true_languages = label_encoder.inverse_transform(test_labels)

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("\nPer-language Accuracy:")
lang_acc = {}
for lang in label_encoder.classes_:
    lang_mask = np.array(true_languages) == lang
    if np.sum(lang_mask) > 0:
        lang_acc[lang] = np.mean(np.array(pred_languages)[lang_mask] == lang)
        print(f"  {lang:15s}: {lang_acc[lang]:.4f} ({np.sum(lang_mask)} samples)")

print("\nClassification Report:")
print(classification_report(
    true_languages,
    pred_languages,
    target_names=label_encoder.classes_,
    zero_division=0
))

if len(label_encoder.classes_) <= 21:
    cm = confusion_matrix(true_languages, pred_languages, labels=label_encoder.classes_)
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_)
    plt.title('AST-GNN Confusion Matrix')
    plt.xlabel('Predicted Language')
    plt.ylabel('True Language')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig('ast_gnn_confusion_matrix.png', dpi=300, bbox_inches='tight')
    print("\nConfusion matrix saved as 'ast_gnn_confusion_matrix.png'")

print("\nSample Predictions:")
print("-" * 40)
for i in range(min(10, len(test_preds))):
    print(f"Sample {i+1}:")
    print(f"  True: {true_languages[i]}")
    print(f"  Pred: {pred_languages[i]}")
    print(f"  Correct: {true_languages[i] == pred_languages[i]}")
    print()

print("\nTraining completed!")
print(f"Best Validation Accuracy: {best_val_acc:.4f}")
print(f"Final Test Accuracy: {test_acc:.4f}")

print("\nSaving complete model for deployment...")

torch.save(model.state_dict(), 'ast_gnn_final_weights.pth')

import pickle
from sklearn.preprocessing import LabelEncoder

checkpoint = {
    'model_state_dict': model.state_dict(),
    'node2idx': node2idx,
    'label_encoder_classes': label_encoder.classes_.tolist(),
    'num_classes': len(tag_vocab),
    'node_input_dim': node_input_dim,
    'edge_input_dim': edge_input_dim,
    'hidden_dim': 128,
    'config': {
        'num_layers': 3,
        'dropout': 0.3
    }
}

import torch.serialization

with torch.serialization.safe_globals([LabelEncoder]):
    torch.save(checkpoint, 'ast_gnn_complete_model.pth')

print("Model saved successfully!")
print("Files created:")
print("  - ast_gnn_final_weights.pth (weights only)")
print("  - ast_gnn_complete_model.pth (complete model)")